# Cleaning Text Data 

The data scraped from RateMyTeacher.com contains more than just reviews. We have factual information about schools and teachers along with review scores. The resulting review corpus is massive so the language modeling analysis will likely benefit from exploring different parts of the corpus separately to see how contexts and topics change. For this we have different **data cleaning settings** that we can try out to review their effect on the outcomes.

**This notebook** will launch spot instances using AWS and clean the data with those spot instances. It will launch an instance for each setting that is attempted. The data configurations can be found in "scripts/cleaning.py" under the `data_configuration_hardcodes` method.

In [7]:
import spotted, os
from IPython.display import clear_output

### Cleaning the Data 

The next two code blocks get everything ready to clean the data. We use bash scripts to run commands on the linux instance, these can be used to create directories in the instance, run scripts, and more. The `cleandata.sh` bash script is what we will use to clean the data but we need to make sure it can be changed to fit each configuration.

***Define* a method to change the cleadata bash script.** Used this to avoid having to feed arguments to `cleandata.sh` in the instance since that requires using `sudo` which is requires changing the settings for python and pip (this is easier).

In [13]:
# A function to modify the cleandata.py bash script to work with whatever argument we pass
def set_cleandata(config):
    
    with open('bash/cleandata.sh', 'r') as f: 
        txt = f.read()
        txt = txt.replace('DC_CONFIG',config)
        txt = txt.replace('DC_OUTFILE','clean_'+config+'.txt')

    with open('bash/cleantemp.sh', 'w') as w: 
        w.write(txt)
        w.close()

**Create spot instances** for each data cleaning configuration and run the setup, and data sleaning scripts. The scripts we run on each instance are: 
- `instancesetup.sh` which creates all the directories necessary
- `script_pull.sh` which pulls the latest version of the scripts directory using git 
- `cleantemp.sh` the version of cleandata.sh specifically for this data configuration

In [11]:
configurations=['A1','B1']
spot_fleet={}

for config in configurations:

    # Modify the cleandata.sh file to run on the current config 
    set_cleandata(config)       

    # Instanciate a connected spot_instance object 
    spot_fleet[config]=spotted.spotted('TEST_'+config, profile='default', filesystem='reviewdata')   
    
    spot_fleet[config].run('bash/instancesetup.sh')     # create the directories and download the libraries needed 
    spot_fleet[config].run('bash/script_pull.sh')       # git pull origin for the scripts directory 
    spot_fleet[config].run('bash/cleantemp.sh')         # run the customized cleandata.sh script (cleantemp.sh)
    
    clear_output() # we'll be left with a lot of connection jargon so we clear it between each instance run 

### Review Outcomes

**Download** logs and the resulting data using a go-between instance

In [8]:
# create and/or connect to an instance you can use to STFP 
cloudwatch=spotted.spotted('cloudwatch', filesystem='reviewdata')
clear_output()
print('cloudwatch instantiated')

cloudwatch instantiated


**List the files** in the directory you're interested in along with their size. Does this match what you expected?

In [10]:
cloudwatch.run('ls efs/data/cleaned_data -l', cmd=True)

>> Connecting.....Connected
total 376576
-rw-rw-r-- 1 ec2-user ec2-user 193020530 Oct  3 05:59 cleaned_docs_A1.pbz2
-rw-rw-r-- 1 ec2-user ec2-user 192588559 Oct  3 06:06 cleaned_docs_B1.pbz2
Time to Run Scripts: 1.7243618965148926


**Specify a list of files** to download and their target local locations

In [12]:
# We want to get the log scripts we created in cleandata.sh 
downloads = ['efs/scripts/scripts/clean_A1.txt','efs/scripts/scripts/clean_B1.txt']

# These will be stored in local_root/archive
localpaths = [os.getcwd()+'/archive/clean_A1.txt',os.getcwd()+'/archive/clean_B1.txt']

# use the download method to get the files
cloudwatch.download(downloads, localpaths)

>> Connecting.....Connected
Time to Download: 4.402894496917725
